# 🏏 Mobile Hawkeye LBW System
### Senior's ball tracking approach adapted for our dataset

**Our dataset structure:**
```
LBW_Dataset/
  lbw/   → VID1_FV.mp4, VID1_SV.mp4, VID5_FV.mp4, VID5_SV.mp4 ...
  nlbw/  → VID2_FV.mp4, VID2_SV.mp4, VID3_FV.mp4, VID3_SV.mp4 ...
```

**Pipeline:**
1. Track ball using color detection (senior's approach)
2. Extract physics features: pitch point, impact point, trajectory, speed
3. Train ML classifier with class balancing
4. Predict LBW for new videos

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1 — Install & Import
# ═══════════════════════════════════════════════════
!pip install -q opencv-python-headless scikit-learn matplotlib seaborn pandas scipy

from google.colab import drive
drive.mount('/content/drive')

import os, cv2, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from tqdm import tqdm
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print('✅ All imports done!')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2 — Paths (UPDATE IF NEEDED)
# ═══════════════════════════════════════════════════

# Our dataset is at:
BASE_PATH  = '/content/drive/MyDrive/LBW_Dataset'
LBW_DIR    = os.path.join(BASE_PATH, 'lbw')    # VID1_FV.mp4, VID1_SV.mp4 ...
NLBW_DIR   = os.path.join(BASE_PATH, 'nlbw')   # VID2_FV.mp4, VID2_SV.mp4 ...
OUTPUT_DIR = '/content/drive/MyDrive/LBW_Hawkeye_Output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify
for d in [LBW_DIR, NLBW_DIR]:
    if os.path.exists(d):
        vids = [f for f in os.listdir(d) if f.endswith('.mp4')]
        sv   = [f for f in vids if '_SV' in f.upper()]
        fv   = [f for f in vids if '_FV' in f.upper()]
        print(f'✅ {os.path.basename(d)}: {len(sv)} SV + {len(fv)} FV videos')
        print(f'   Sample SV: {sv[:3]}')
        print(f'   Sample FV: {fv[:3]}')
    else:
        print(f'❌ NOT FOUND: {d}')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3 — Ball Tracking (Senior's approach)
# Tracks red cricket ball using HSV color detection
# ═══════════════════════════════════════════════════

def track_ball(video_path):
    """
    Senior's ball tracking function.
    Tracks cricket ball frame by frame using color segmentation.
    Returns array of (frame_num, x, y) positions.
    """
    cap        = cv2.VideoCapture(video_path)
    trajectory = []
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1

        blurred = cv2.GaussianBlur(frame, (11, 11), 0)
        hsv     = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)

        # Red ball (senior's original range)
        lower_red = np.array([0,  100, 100])
        upper_red = np.array([10, 255, 255])
        mask = cv2.inRange(hsv, lower_red, upper_red)

        # Also try red on other side of hue wheel
        lower_red2 = np.array([160, 100, 100])
        upper_red2 = np.array([180, 255, 255])
        mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
        mask  = cv2.bitwise_or(mask, mask2)

        # Also try white ball
        lower_white = np.array([0, 0, 200])
        upper_white = np.array([180, 30, 255])
        mask3 = cv2.inRange(hsv, lower_white, upper_white)
        mask  = cv2.bitwise_or(mask, mask3)

        # Also try orange/yellow (tennis ball)
        lower_yellow = np.array([15, 100, 100])
        upper_yellow = np.array([35, 255, 255])
        mask4 = cv2.inRange(hsv, lower_yellow, upper_yellow)
        mask  = cv2.bitwise_or(mask, mask4)

        # Morphological cleanup
        mask = cv2.erode(mask,  None, iterations=1)
        mask = cv2.dilate(mask, None, iterations=2)

        contours, _ = cv2.findContours(
            mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        if contours:
            largest    = max(contours, key=cv2.contourArea)
            (x, y), r  = cv2.minEnclosingCircle(largest)
            if r > 2:   # filter noise
                trajectory.append((frame_count, int(x), int(y)))

    cap.release()
    return np.array(trajectory) if len(trajectory) > 0 else np.array([]).reshape(0, 3)


# Test on one video
test_sv = [f for f in os.listdir(LBW_DIR) if '_SV' in f.upper()][0]
traj    = track_ball(os.path.join(LBW_DIR, test_sv))
print(f'Test video : {test_sv}')
print(f'Ball points detected: {len(traj)}')
if len(traj) > 0:
    print('✅ Ball tracking working!')
else:
    print('⚠️  Ball not detected — will use zero features as fallback')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 4 — Feature Extractors (Senior's approach)
# ═══════════════════════════════════════════════════

def fit_trajectory(trajectory):
    """Fit parabola to ball trajectory (senior's function)."""
    def parabola(x, a, b, c):
        return a * x**2 + b * x + c
    try:
        x      = trajectory[:, 1].astype(float)
        y      = trajectory[:, 2].astype(float)
        params, _ = curve_fit(parabola, x, y, maxfev=10000)
        return params, parabola
    except:
        return None, None


def detect_pitch_point(trajectory):
    """Find where ball bounces (senior's function)."""
    y  = trajectory[:, 2]
    dy = np.diff(y)
    return trajectory[np.argmin(dy)]


def detect_impact_point(trajectory):
    """Find where ball hits pad — largest direction change (senior's function)."""
    dx     = np.diff(trajectory[:, 1].astype(float))
    dy     = np.diff(trajectory[:, 2].astype(float))
    change = np.sqrt(dx**2 + dy**2)
    return trajectory[np.argmax(change)]


def estimate_speed(trajectory, fps=30):
    """Estimate ball speed (senior's function)."""
    dist = np.sum(np.sqrt(
        np.diff(trajectory[:, 1].astype(float))**2 +
        np.diff(trajectory[:, 2].astype(float))**2
    ))
    return dist / (len(trajectory) / fps)


def normalize_height(y, frame_height=480):
    """Normalize y coordinate by frame height (senior's function)."""
    return y / frame_height


def is_in_line(x, frame_width=640):
    """
    Check if ball is in line with stumps.
    Senior used fixed range (300,340) — we use dynamic based on frame width.
    Middle 15% of frame = stump zone.
    """
    stump_left  = frame_width * 0.42
    stump_right = frame_width * 0.58
    return int(stump_left <= x <= stump_right)


def get_frame_dims(video_path):
    cap = cv2.VideoCapture(video_path)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return w, h


def extract_features(side_path, front_path):
    """
    Senior's extract_features adapted for our dataset.
    Uses side view for trajectory/physics, front view for line check.
    """
    side_traj  = track_ball(side_path)
    front_traj = track_ball(front_path)

    sw, sh = get_frame_dims(side_path)
    fw, fh = get_frame_dims(front_path)

    # If ball not detected in either view — return zero features
    if len(side_traj) < 5 and len(front_traj) < 5:
        return {
            'ball_pitch_x': 0, 'ball_pitch_y': 0,
            'impact_x': 0, 'impact_y': 0,
            'speed': 0,
            'trajectory_hits_stumps': 0,
            'impact_in_line': 0,
            'impact_height_norm': 0,
            'sv_x_range': 0, 'sv_y_range': 0,
            'sv_traj_len': 0, 'fv_traj_len': 0,
            'tracking_quality': 0
        }

    features = {}

    # ── Side view features ──────────────────────────
    if len(side_traj) >= 5:
        parab_params, parab_func = fit_trajectory(side_traj)
        bounce       = detect_pitch_point(side_traj)
        impact       = detect_impact_point(side_traj)
        speed        = estimate_speed(side_traj)

        features['ball_pitch_x']      = bounce[1] / sw
        features['ball_pitch_y']      = bounce[2] / sh
        features['impact_x']          = impact[1] / sw
        features['impact_y']          = impact[2] / sh
        features['speed']             = speed
        features['impact_height_norm']= normalize_height(impact[2], sh)
        features['sv_x_range']        = (side_traj[:,1].max() - side_traj[:,1].min()) / sw
        features['sv_y_range']        = (side_traj[:,2].max() - side_traj[:,2].min()) / sh
        features['sv_traj_len']       = len(side_traj)

        # Does trajectory hit stump height after impact?
        if parab_params is not None:
            try:
                future_y = parab_func(impact[1] + 50, *parab_params)
                features['trajectory_hits_stumps'] = int(future_y < sh * 0.55)
            except:
                features['trajectory_hits_stumps'] = 0
        else:
            features['trajectory_hits_stumps'] = 0
    else:
        features.update({
            'ball_pitch_x': 0, 'ball_pitch_y': 0,
            'impact_x': 0,     'impact_y': 0,
            'speed': 0,        'impact_height_norm': 0,
            'sv_x_range': 0,   'sv_y_range': 0,
            'sv_traj_len': 0,  'trajectory_hits_stumps': 0
        })

    # ── Front view features ─────────────────────────
    if len(front_traj) >= 5:
        front_impact = detect_impact_point(front_traj)
        features['impact_in_line'] = is_in_line(front_impact[1], fw)
        features['fv_traj_len']    = len(front_traj)
    else:
        features['impact_in_line'] = 0
        features['fv_traj_len']    = 0

    features['tracking_quality'] = int(len(side_traj)>=5) + int(len(front_traj)>=5)
    return features


print('✅ Feature extraction functions ready!')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 5 — Process All Videos & Build CSV
# Adapted from senior's process_all_videos()
# ═══════════════════════════════════════════════════

def process_all_videos(lbw_dir, nlbw_dir):
    """
    Adapted from senior's code for our dataset structure.
    Our files: VID1_FV.mp4, VID1_SV.mp4 (not sv_1.mp4, fv_1.mp4)
    """
    final_data = []

    for folder, label, name in [
        (nlbw_dir, 0, 'Not_LBW'),
        (lbw_dir,  1, 'LBW')
    ]:
        all_files = sorted(os.listdir(folder))
        # Get SV files (side view)
        sv_files  = [f for f in all_files
                     if f.endswith('.mp4') and '_SV' in f.upper()]

        print(f'\n📂 {name} — {len(sv_files)} SV videos')

        for sv_file in tqdm(sv_files, desc=name):
            # Find matching FV file
            # e.g. VID1_SV.mp4 → look for VID1_FV.mp4
            vid_id  = Path(sv_file).stem.upper().replace('_SV', '')
            fv_file = next(
                (f for f in all_files
                 if vid_id in f.upper() and '_FV' in f.upper()),
                None
            )

            if fv_file is None:
                print(f'  ❌ Missing FV for {sv_file}, skipping')
                continue

            sv_path = os.path.join(folder, sv_file)
            fv_path = os.path.join(folder, fv_file)

            print(f'  Processing: {sv_file} + {fv_file}')

            # Track ball
            sv_traj = track_ball(sv_path)
            fv_traj = track_ball(fv_path)

            if len(sv_traj) < 5:
                print(f'  ⚠️  SV: only {len(sv_traj)} points (using zeros)')
            if len(fv_traj) < 5:
                print(f'  ⚠️  FV: only {len(fv_traj)} points (using zeros)')

            features = extract_features(sv_path, fv_path)
            if features is not None:
                features['label']    = label
                features['video_id'] = vid_id
                features['source']   = name
                final_data.append(features)
            else:
                print(f'  ⚠️  Feature extraction failed for {sv_file}')

    df = pd.DataFrame(final_data)
    csv_path = os.path.join(OUTPUT_DIR, 'final_lbw_dataset.csv')
    df.to_csv(csv_path, index=False)

    print(f'\n📊 Dataset Summary:')
    print(f'   Total   : {len(df)} videos processed')
    print(f'   LBW     : {(df.label==1).sum()}')
    print(f'   Not LBW : {(df.label==0).sum()}')
    print(f'   Ball tracked both views: {(df.tracking_quality==2).sum()}/{len(df)}')
    print(f'\n✅ Saved to {csv_path}')
    return df


print('🔍 Processing all videos (this takes 5-10 minutes)...')
df = process_all_videos(LBW_DIR, NLBW_DIR)
print(df[['video_id','source','tracking_quality','trajectory_hits_stumps','impact_in_line']].to_string())

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 6 — Visualise Feature Distributions
# ═══════════════════════════════════════════════════
feat_cols = [c for c in df.columns
             if c not in ['label','video_id','source']]

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i, col in enumerate(feat_cols[:12]):
    ax = axes[i//4][i%4]
    lbw_v  = df[df.label==1][col].dropna()
    nlbw_v = df[df.label==0][col].dropna()
    ax.hist(lbw_v,  bins=10, alpha=0.6, color='red',   label='LBW')
    ax.hist(nlbw_v, bins=10, alpha=0.6, color='green', label='Not LBW')
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)
plt.suptitle('Feature Distributions: LBW vs Not LBW', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_distributions.png'), dpi=120)
plt.show()

# Correlation
plt.figure(figsize=(10, 7))
sns.heatmap(df[feat_cols+['label']].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Feature Correlation with LBW Label', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 7 — Train ML Classifier
# Random Forest + SVM + Gradient Boost ensemble
# class_weight='balanced' fixes 11 LBW vs 56 NLBW
# ═══════════════════════════════════════════════════
feat_cols = [c for c in df.columns
             if c not in ['label','video_id','source']]

X = df[feat_cols].fillna(0).values
y = df['label'].values

print(f'Feature matrix : {X.shape}')
print(f'LBW samples    : {y.sum()}')
print(f'Not LBW samples: {(y==0).sum()}')

# Scale features
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split — stratified to keep class ratio
X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y, test_size=0.25,
    stratify=y, random_state=42
)

# Classifiers with class balancing
rf  = RandomForestClassifier(
    n_estimators=300, class_weight='balanced',
    max_depth=8, random_state=42
)
gb  = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05,
    max_depth=4, random_state=42
)
svm = SVC(
    probability=True, class_weight='balanced',
    kernel='rbf', random_state=42
)

# Ensemble
ens = VotingClassifier(
    estimators=[('rf',rf),('gb',gb),('svm',svm)],
    voting='soft'
)

# 5-fold cross validation
print('\n🔄 5-fold Cross-Validation (macro F1):')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, clf in [('Random Forest',rf),('Grad Boost',gb),
                  ('SVM',svm),('Ensemble',ens)]:
    s = cross_val_score(clf, X_scaled, y, cv=cv, scoring='f1_macro')
    print(f'  {name:15s}: {s.mean():.3f} +/- {s.std():.3f}')

# Train final model
print('\n🚀 Training final ensemble...')
ens.fit(X_tr, y_tr)

# Evaluate
y_pred = ens.predict(X_te)
y_prob = ens.predict_proba(X_te)[:, 1]

print('\n📊 Test Results:')
print(classification_report(y_te, y_pred,
      target_names=['Not LBW', 'LBW']))
try:
    print(f'AUC: {roc_auc_score(y_te, y_prob):.4f}')
except: pass

# Confusion matrix
cm = confusion_matrix(y_te, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not LBW','LBW'],
            yticklabels=['Not LBW','LBW'])
plt.title('Confusion Matrix', fontweight='bold')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 8 — Feature Importance & Save Model
# ═══════════════════════════════════════════════════
rf.fit(X_tr, y_tr)
imp = pd.Series(rf.feature_importances_,
                index=feat_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 4))
imp.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Feature Importance (Random Forest)', fontweight='bold')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=150)
plt.show()

print('Top features that determine LBW:')
print(imp.head(8).to_string())

# Save model
with open(os.path.join(OUTPUT_DIR,'lbw_classifier.pkl'),'wb') as f: pickle.dump(ens,    f)
with open(os.path.join(OUTPUT_DIR,'lbw_scaler.pkl'),   'wb') as f: pickle.dump(scaler,  f)
with open(os.path.join(OUTPUT_DIR,'feat_cols.pkl'),    'wb') as f: pickle.dump(feat_cols,f)

# Verify saved
for fname in ['lbw_classifier.pkl','lbw_scaler.pkl','feat_cols.pkl']:
    p = os.path.join(OUTPUT_DIR, fname)
    print(f'  ✅ {fname} — {os.path.getsize(p)/1024:.1f} KB')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 9 — PREDICTION FUNCTION
# Give FV + SV video → get LBW decision
# Shows ball trajectory on frame
# ═══════════════════════════════════════════════════

def predict_lbw(fv_video_path, sv_video_path):
    """
    Predict LBW decision for a new delivery.
    Args:
        fv_video_path : front view video
        sv_video_path : side view video
    """
    print(f'\n🏏 Analysing delivery...')
    print(f'   FV: {os.path.basename(fv_video_path)}')
    print(f'   SV: {os.path.basename(sv_video_path)}')

    # Extract features
    feat = extract_features(sv_video_path, fv_video_path)
    tq   = feat['tracking_quality']

    if tq == 0:
        print('  ⚠️  Ball not detected in either view!')
        print('  Result may be inaccurate.')
    elif tq == 1:
        print('  ⚠️  Ball detected in only one view.')
    else:
        print('  ✅ Ball tracked in both views.')

    # Predict
    fd      = pd.DataFrame([feat])[feat_cols].fillna(0)
    prob    = ens.predict_proba(scaler.transform(fd.values))[0][1]
    verdict = '🔴 OUT — LBW!' if prob >= 0.5 else '🟢 NOT OUT'

    # Get middle frame from each video
    def get_frame(path, pos=0.5):
        cap = cv2.VideoCapture(path)
        n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * pos))
        ret, frame = cap.read()
        cap.release()
        if ret:
            return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        return np.zeros((480, 640, 3), dtype=np.uint8)

    # Draw ball trajectory on frame
    def draw_trajectory(frame_rgb, traj, color=(255, 255, 0)):
        vis = frame_rgb.copy()
        for i in range(1, len(traj)):
            p1 = (int(traj[i-1][1]), int(traj[i-1][2]))
            p2 = (int(traj[i][1]),   int(traj[i][2]))
            cv2.line(vis, p1, p2, color, 2)
            cv2.circle(vis, p2, 5, (255, 50, 50), -1)
        return vis

    sv_traj = track_ball(sv_video_path)
    fv_traj = track_ball(fv_video_path)

    sv_frame = get_frame(sv_video_path)
    fv_frame = get_frame(fv_video_path)

    sv_vis = draw_trajectory(sv_frame, sv_traj) if len(sv_traj) > 1 else sv_frame
    fv_vis = draw_trajectory(fv_frame, fv_traj) if len(fv_traj) > 1 else fv_frame

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    axes[0].imshow(fv_vis)
    axes[0].set_title('Front View\n(yellow = ball trajectory)', fontsize=11)
    axes[0].axis('off')

    axes[1].imshow(sv_vis)
    axes[1].set_title('Side View\n(yellow = ball trajectory)', fontsize=11)
    axes[1].axis('off')

    # Decision panel
    color = '#FF1744' if prob >= 0.5 else '#00C853'
    axes[2].set_facecolor('#F0F0F0')
    axes[2].text(0.5, 0.78, verdict,
                 ha='center', fontsize=17, fontweight='bold',
                 color=color, transform=axes[2].transAxes)
    axes[2].text(0.5, 0.58, f'LBW Probability : {prob:.1%}',
                 ha='center', fontsize=12, transform=axes[2].transAxes)
    axes[2].text(0.5, 0.42,
                 f'In Line      : {"✅ YES" if feat["impact_in_line"] else "❌ NO"}',
                 ha='center', fontsize=11, transform=axes[2].transAxes)
    axes[2].text(0.5, 0.28,
                 f'Hits Stumps  : {"✅ YES" if feat["trajectory_hits_stumps"] else "❌ NO"}',
                 ha='center', fontsize=11, transform=axes[2].transAxes)
    axes[2].text(0.5, 0.13,
                 f'Tracking     : {"✅ Both" if tq==2 else "⚠️ Partial"}',
                 ha='center', fontsize=10, color='gray',
                 transform=axes[2].transAxes)
    axes[2].axis('off')
    axes[2].set_title('⚖️ Hawkeye Decision', fontsize=12, fontweight='bold')

    plt.suptitle('🏏 Mobile Hawkeye LBW System', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'last_prediction.png'), dpi=150)
    plt.show()

    print(f'\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print(f'  LBW Probability  : {prob:.4f}')
    print(f'  In Line          : {bool(feat["impact_in_line"])}')
    print(f'  Hits Stumps      : {bool(feat["trajectory_hits_stumps"])}')
    print(f'  Ball Tracking    : {tq}/2 views')
    print(f'  DECISION         : {verdict}')
    print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    return verdict, prob


print('✅ predict_lbw() is ready!')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 10 — RUN PREDICTION
# Change paths to your test videos
# ═══════════════════════════════════════════════════

# ── Option A: Test on your uploaded videos ──────────────────
predict_lbw(
    fv_video_path='/content/drive/MyDrive/FV.mp4',
    sv_video_path='/content/drive/MyDrive/SV.mp4'
)

In [ ]:
# ── Option B: Test on LBW dataset video ─────────────────────
# predict_lbw(
#     fv_video_path='/content/drive/MyDrive/LBW_Dataset/lbw/VID1_FV.mp4',
#     sv_video_path='/content/drive/MyDrive/LBW_Dataset/lbw/VID1_SV.mp4'
# )

# ── Option C: Test on Not-LBW dataset video ──────────────────
# predict_lbw(
#     fv_video_path='/content/drive/MyDrive/LBW_Dataset/nlbw/VID2_FV.mp4',
#     sv_video_path='/content/drive/MyDrive/LBW_Dataset/nlbw/VID2_SV.mp4'
# )
print('Uncomment one of the options above to test!')